In [ ]:
import numpy as np
import scipy.io as sio
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import math
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load data
# In a real Jupyter environment, you'd place 'data.mat' and 'label.mat' in the same directory as your notebook.
# For this simulation, we'll assume they are accessible.
try:
    mat_data = sio.loadmat('data.mat')
    mat_label = sio.loadmat('label.mat')
    images = mat_data['data']
    labels = mat_label['trueLabel'][0]
    print("Data loaded successfully from .mat files.")
except FileNotFoundError:
    print("Error: 'data.mat' or 'label.mat' not found. Please ensure they are in the same directory.")
    print("Generating dummy data for demonstration purposes.")
    # Generate dummy data with similar structure for demonstration
    num_images_dummy = 1990
    image_dim_dummy = 784
    images = np.random.rand(image_dim_dummy, num_images_dummy) * 255 # Simulate pixel values
    labels = np.random.choice([2, 6, 1, 3, 4, 5, 7, 8, 9, 0], num_images_dummy) # Simulate labels
    print("Dummy data generated.")

# Filter for digits '2' and '6'
idx_2_6 = np.where((labels == 2) | (labels == 6))[0]
images_2_6 = images[:, idx_2_6]
labels_2_6 = labels[idx_2_6]

print(f"Original images shape: {images.shape}")
print(f"Filtered images shape (2s and 6s): {images_2_6.shape}")
print(f"Filtered labels shape (2s and 6s): {labels_2_6.shape}")

In [ ]:
# Transpose images_2_6 so that rows are samples and columns are features
data_2_6 = images_2_6.T

# Apply PCA to reduce dimensionality to 4
n_components = 4
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(data_2_6)

print(f"Data shape after PCA: {X_pca.shape}")
print(f"Explained variance ratio by principal components: {pca.explained_variance_ratio_}")
print(f"Sum of explained variance ratio: {np.sum(pca.explained_variance_ratio_):.4f}")

In [ ]:
def gaussian_pdf(X, mean, cov):
    """
    Calculates the probability density function of a multivariate Gaussian distribution.
    X: Data points (N_samples, N_dimensions)
    mean: Mean vector (N_dimensions,)
    cov: Covariance matrix (N_dimensions, N_dimensions)
    """
    N = X.shape[0]
    D = X.shape[1]
    
    # Add a small value to the diagonal of the covariance matrix for numerical stability
    # This helps in cases where the covariance matrix might become singular.
    cov = cov + np.eye(D) * 1e-6 
    
    det_cov = np.linalg.det(cov)
    if det_cov == 0:
        # Handle singular matrix case - this should ideally be avoided with regularization
        # or by checking for positive definiteness during M-step.
        # For now, return a very small probability to penalize it.
        return np.full(N, 1e-100) # A very small non-zero probability
    
    inv_cov = np.linalg.inv(cov)
    
    # Reshape mean for broadcasting
    mean_reshaped = mean.reshape(1, -1)

    X_minus_mean = X - mean_reshaped
    
    # Calculate the exponent term: -0.5 * (X - mu)T * Sigma^-1 * (X - mu)
    # Using Einstein summation for efficient matrix multiplication
    exponent = -0.5 * np.einsum('ij,jk,ik->i', X_minus_mean, inv_cov, X_minus_mean)
    
    # Calculate the coefficient term: 1 / sqrt((2*pi)^D * det(Sigma))
    coefficient = 1 / np.sqrt((2 * np.pi)**D * det_cov)
    
    return coefficient * np.exp(exponent)

def calculate_log_likelihood(X, weights, means, covariances, C):
    """
    Calculates the log-likelihood for the GMM.
    X: Data points (N_samples, N_dimensions)
    weights: Component weights (C,)
    means: Component means (C, N_dimensions)
    covariances: Component covariance matrices (C, N_dimensions, N_dimensions)
    C: Number of components
    """
    N = X.shape[0]
    log_likelihood = 0.0
    
    for i in range(N):
        sum_components = 0.0
        for k in range(C):
            # gaussian_pdf returns an array, even for a single sample.
            # We need to extract the scalar value from it.
            pdf_val = gaussian_pdf(X[i:i+1, :], means[k], covariances[k])
            sum_components += weights[k] * pdf_val[0] # Access the scalar value using [0]
        
        # Add a small epsilon to sum_components to prevent log(0) if it rarely happens
        log_likelihood += np.log(sum_components + 1e-300) 
        
    return log_likelihood

In [ ]:
def em_gmm(X, C, max_iter=100, tol=1e-4):
    """
    Implements the EM algorithm for Gaussian Mixture Model.
    X: Data points (N_samples, N_dimensions)
    C: Number of components (clusters)
    max_iter: Maximum number of iterations
    tol: Tolerance for convergence (change in log-likelihood)
    """
    N, D = X.shape

    # 1. Initialization
    # Initialize weights uniformly
    weights = np.ones(C) / C

    # Initialize means: random Gaussian vector with zero mean
    means = np.random.randn(C, D) * 0.1 # Small random values

    # Initialize covariances: S_k S_k^T + I_D
    covariances = np.zeros((C, D, D))
    for k in range(C):
        S_k = np.random.randn(D, D) # Gaussian random matrix
        covariances[k] = S_k @ S_k.T + np.eye(D)
    
    log_likelihoods = []
    prev_log_likelihood = -np.inf

    for iteration in range(max_iter):
        # E-Step: Calculate responsibilities (tau_k_i)
        tau_k_i = np.zeros((N, C))
        for k in range(C):
            pdf_values = gaussian_pdf(X, means[k], covariances[k])
            tau_k_i[:, k] = weights[k] * pdf_values

        # Normalize responsibilities for each data point
        sum_tau = np.sum(tau_k_i, axis=1, keepdims=True)
        # Handle cases where sum_tau might be zero (very low probability data point for all components)
        # Add a small epsilon to avoid division by zero
        tau_k_i /= (sum_tau + 1e-300) 

        # M-Step: Update parameters
        # Update weights
        N_k = np.sum(tau_k_i, axis=0)
        weights = N_k / N

        # Update means
        for k in range(C):
            means[k] = np.sum(tau_k_i[:, k, np.newaxis] * X, axis=0) / (N_k[k] + 1e-300)

        # Update covariances
        for k in range(C):
            X_minus_mean_k = X - means[k]
            # Reshape tau_k_i[:, k] to (N, 1) for broadcasting
            covariances[k] = (X_minus_mean_k.T @ (tau_k_i[:, k, np.newaxis] * X_minus_mean_k)) / (N_k[k] + 1e-300)
            # Add a small value to the diagonal for numerical stability (regularization)
            covariances[k] += np.eye(D) * 1e-6 

        # Calculate log-likelihood
        current_log_likelihood = calculate_log_likelihood(X, weights, means, covariances, C)
        log_likelihoods.append(current_log_likelihood)

        # Check for convergence
        if iteration > 0 and abs(current_log_likelihood - prev_log_likelihood) < tol:
            print(f"EM converged at iteration {iteration}")
            break
        prev_log_likelihood = current_log_likelihood
        
        # Optional: Print progress
        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(f"Iteration {iteration+1}, Log-Likelihood: {current_log_likelihood:.4f}")

    return weights, means, covariances, tau_k_i, log_likelihoods

In [ ]:
# Run EM algorithm
C = 2 # Number of components for GMM (for digits '2' and '6')
weights, means_pca, covariances_pca, responsibilities, log_likelihoods = em_gmm(X_pca, C)

# Plot log-likelihood
plt.figure(figsize=(10, 6))
plt.plot(range(len(log_likelihoods)), log_likelihoods, marker='o', linestyle='-')
plt.title('Log-Likelihood vs. Number of Iterations')
plt.xlabel('Iteration')
plt.ylabel('Log-Likelihood')
plt.grid(True)
plt.show()

In [ ]:
# Part 2: Report Fitted GMM Model

print("--- Fitted GMM Model Parameters ---")

# a) Numerical weights for each component
print("\n1. Component Weights:")
for i, w in enumerate(weights):
    print(f"   Component {i+1}: {w:.4f}")

# b) Mean of each component (mapped back to original space and as 28x28 images)
print("\n2. Component Means (as 28x28 Images):")

# Map means back to original 784-dimensional space
# The inverse transform requires the components and mean from PCA
# X_original_recon = pca.inverse_transform(X_pca) would give the full data reconstruction
# To reconstruct the mean of a component, we use pca.inverse_transform on the mean vector
# Note: The pca.mean_ attribute is crucial here for correct reconstruction
original_space_means = pca.inverse_transform(means_pca)

plt.figure(figsize=(10, 5))
for i, mean_vec in enumerate(original_space_means):
    # Reshape the 784-dimensional vector to 28x28 image
    mean_image = mean_vec.reshape(28, 28)
    
    plt.subplot(1, C, i + 1)
    plt.imshow(mean_image, cmap='gray')
    plt.title(f'Component {i+1} Mean (Digit {2 if i==0 else 6})' if labels_2_6[responsibilities[:, i].argmax()] == 2 or labels_2_6[responsibilities[:, i].argmax()] == 6 else f'Component {i+1} Mean')
    plt.axis('off')
plt.suptitle('Mean Images of GMM Components (in Original 784-dim Space)')
plt.show()


# c) Two 4x4 covariance matrices (visualized as heatmaps)
print("\n3. Covariance Matrices (4x4 Heatmaps):")
plt.figure(figsize=(12, 5))
for i, cov_mat in enumerate(covariances_pca):
    plt.subplot(1, C, i + 1)
    plt.imshow(cov_mat, cmap='viridis', interpolation='nearest')
    plt.colorbar(label='Covariance Value')
    plt.title(f'Component {i+1} Covariance Matrix')
    plt.xlabel('Dimension')
    plt.ylabel('Dimension')
plt.suptitle('Covariance Matrices of GMM Components (in 4-dim PCA Space)')
plt.show()

print("\nNumerical Covariance Matrices:")
for i, cov_mat in enumerate(covariances_pca):
    print(f"\nComponent {i+1} Covariance Matrix:\n{cov_mat.round(4)}")

In [ ]:
# Part 3: Infer Labels and Compare with True Labels

# Determine which GMM component corresponds to '2' and '6'
# We assign the component label based on the majority true label of the data points assigned to that component.
cluster_labels_map = {}
for k in range(C):
    # Find the indices of data points primarily assigned to component k
    component_k_indices = np.where(np.argmax(responsibilities, axis=1) == k)[0]
    if len(component_k_indices) > 0:
        # Get the true labels for these data points
        true_labels_in_component_k = labels_2_6[component_k_indices]
        # Count occurrences of '2' and '6'
        count_2 = np.sum(true_labels_in_component_k == 2)
        count_6 = np.sum(true_labels_in_component_k == 6)
        
        # Assign the cluster label based on the majority
        if count_2 > count_6:
            cluster_labels_map[k] = 2
        else:
            cluster_labels_map[k] = 6
    else:
        # If no data points are assigned, assign a default or handle as an edge case
        cluster_labels_map[k] = -1 # Or some other indicator

print("GMM Cluster to True Label Mapping:", cluster_labels_map)

# Infer GMM labels for all data points
gmm_predicted_raw_labels = np.argmax(responsibilities, axis=1)
gmm_predicted_labels = np.array([cluster_labels_map[label] for label in gmm_predicted_raw_labels])

# Calculate overall accuracy and mis-classification rate for GMM
correct_predictions_gmm = np.sum(gmm_predicted_labels == labels_2_6)
total_predictions = len(labels_2_6)
accuracy_gmm = correct_predictions_gmm / total_predictions
misclassification_rate_gmm = 1 - accuracy_gmm

print(f"\n--- GMM Performance ---")
print(f"GMM Overall Accuracy: {accuracy_gmm:.4f}")
print(f"GMM Overall Mis-classification Rate: {misclassification_rate_gmm:.4f}")

# Calculate mis-classification rate for '2' and '6' separately for GMM
mis_2_gmm = np.sum((gmm_predicted_labels != 2) & (labels_2_6 == 2)) / np.sum(labels_2_6 == 2)
mis_6_gmm = np.sum((gmm_predicted_labels != 6) & (labels_2_6 == 6)) / np.sum(labels_2_6 == 6)

print(f"GMM Mis-classification Rate for Digit '2': {mis_2_gmm:.4f}")
print(f"GMM Mis-classification Rate for Digit '6': {mis_6_gmm:.4f}")


# Perform K-Means clustering with K=2
print("\n--- K-Means Clustering Performance ---")
kmeans = KMeans(n_clusters=C, random_state=0, n_init=10) # n_init for robust initialization
kmeans_labels_raw = kmeans.fit_predict(X_pca)

# Map K-Means clusters to true labels ('2' or '6')
kmeans_cluster_labels_map = {}
for k in range(C):
    component_k_indices = np.where(kmeans_labels_raw == k)[0]
    if len(component_k_indices) > 0:
        true_labels_in_component_k = labels_2_6[component_k_indices]
        count_2 = np.sum(true_labels_in_component_k == 2)
        count_6 = np.sum(true_labels_in_component_k == 6)
        if count_2 > count_6:
            kmeans_cluster_labels_map[k] = 2
        else:
            kmeans_cluster_labels_map[k] = 6
    else:
        kmeans_cluster_labels_map[k] = -1

print("K-Means Cluster to True Label Mapping:", kmeans_cluster_labels_map)

kmeans_predicted_labels = np.array([kmeans_cluster_labels_map[label] for label in kmeans_labels_raw])

# Calculate overall accuracy and mis-classification rate for K-Means
correct_predictions_kmeans = np.sum(kmeans_predicted_labels == labels_2_6)
accuracy_kmeans = correct_predictions_kmeans / total_predictions
misclassification_rate_kmeans = 1 - accuracy_kmeans

print(f"K-Means Overall Accuracy: {accuracy_kmeans:.4f}")
print(f"K-Means Overall Mis-classification Rate: {misclassification_rate_kmeans:.4f}")

# Calculate mis-classification rate for '2' and '6' separately for K-Means
mis_2_kmeans = np.sum((kmeans_predicted_labels != 2) & (labels_2_6 == 2)) / np.sum(labels_2_6 == 2)
mis_6_kmeans = np.sum((kmeans_predicted_labels != 6) & (labels_2_6 == 6)) / np.sum(labels_2_6 == 6)

print(f"K-Means Mis-classification Rate for Digit '2': {mis_2_kmeans:.4f}")
print(f"K-Means Mis-classification Rate for Digit '6': {mis_6_kmeans:.4f}")

print("\n--- Comparison ---")
if misclassification_rate_gmm < misclassification_rate_kmeans:
    print("GMM achieves better overall performance (lower mis-classification rate).")
elif misclassification_rate_gmm > misclassification_rate_kmeans:
    print("K-Means achieves better overall performance (lower mis-classification rate).")
else:
    print("GMM and K-Means achieve similar overall performance.")